# Experiment 08 — REST API Workflow

Curl examples for the Polomni lab API. Start the server in a separate terminal
before running these commands:

```bash
cd deepiri-polomni
poetry run polomni serve --host 127.0.0.1 --port 8091
```

Base URL: `http://127.0.0.1:8091`

## Health checks

```bash
# Liveness probe
curl -s http://127.0.0.1:8091/health | jq .

# Service metadata
curl -s http://127.0.0.1:8091/ | jq .
```

Expected `/health` response:

```json
{"status": "ok", "service": "polomni-lab"}
```

## Data catalog and cache status

```bash
# List fetchable products
curl -s http://127.0.0.1:8091/data/catalog | jq '.[].id'

# Show which products are cached locally
curl -s http://127.0.0.1:8091/data/status | jq '.[] | {product_id, cached, size_bytes}'
```

## Fetch lite products

```bash
curl -s -X POST http://127.0.0.1:8091/data/fetch \
  -H 'Content-Type: application/json' \
  -d '{
    "product_ids": ["planck_cmb_tt_power", "planck_lcdm_baseline"],
    "force": false,
    "fetch_gw": true
  }' | jq .
```

Response includes `results[]` per product and `gw_events_count` when GW refresh is enabled.

## GW event summary

```bash
# Requires GWTC cached via POST /data/fetch first
curl -s http://127.0.0.1:8091/data/gw/events | jq '{count, recent: .recent[:3]}'
```

## Observatory scan (synthetic smoke)

Fast scan with no network or FITS dependencies:

```bash
curl -s -X POST http://127.0.0.1:8091/observatory/scan \
  -H 'Content-Type: application/json' \
  -d '{"synthetic": true, "nside": 32, "nulls": 5, "seed": 42}' | jq .
```

## Observatory scan (cached WMAP map)

Requires WMAP K-band in cache (fetch via CLI or REST first):

```bash
curl -s -X POST http://127.0.0.1:8091/data/fetch \
  -H 'Content-Type: application/json' \
  -d '{"product_ids": ["wmap_k_band"], "fetch_gw": false}'

curl -s -X POST http://127.0.0.1:8091/observatory/scan \
  -H 'Content-Type: application/json' \
  -d '{"synthetic": false, "map_product_id": "wmap_k_band", "nside": 64, "nulls": 10}' | jq .
```

## Full pipeline

Ingest → load → downsample → RBLE score → JSON report:

```bash
curl -s -X POST http://127.0.0.1:8091/observatory/pipeline \
  -H 'Content-Type: application/json' \
  -d '{"map_product": "wmap_k_band", "nside": 64, "nulls": 10, "force": false}' | jq .
```

## List detection reports

```bash
curl -s http://127.0.0.1:8091/observatory/reports | jq '.reports[:5]'
```

## SSE — GW poll stream

Server-Sent Events stream (use `-N` to disable curl buffering):

```bash
curl -N 'http://127.0.0.1:8091/stream/gw/poll?interval=2&max_events=3'
```

Each line is an SSE `data:` frame with JSON:

```json
{"timestamp": "...", "kind": "gw_poll", "results_count": 93, "new_events": 0, "message": "..."}
```

## Full workflow script

Copy-paste end-to-end API session:

```bash
BASE=http://127.0.0.1:8091

curl -s "$BASE/health" | jq .
curl -s "$BASE/data/catalog" | jq 'length'
curl -s -X POST "$BASE/data/fetch" \
  -H 'Content-Type: application/json' \
  -d '{"product_ids":["planck_cmb_tt_power"],"fetch_gw":true}' | jq .
curl -s -X POST "$BASE/observatory/scan" \
  -H 'Content-Type: application/json' \
  -d '{"synthetic":true,"nside":32,"nulls":5}' | jq '{rble_score, null_sigma}'
curl -s "$BASE/observatory/reports" | jq '.reports | length'
```

See also: [docs/guides/api_reference.md](../docs/guides/api_reference.md)